# 01 - Fundamentos de Clustering Jerárquico Aglomerativo y Dendrogramas

## Curso de Machine Learning: Modelos No Supervisados

En los módulos anteriores analizamos dos paradigmas de agrupamiento:
1. **K-Means**: Basado en centroides y minimización de varianza intra-cluster (requiere definir $K$ a priori y asume esfericidad).
2. **DBSCAN**: Basado en densidad local y conectividad espacial (inmune a outliers pero dependiente de $\varepsilon$).

El **Agrupamiento Jerárquico (*Hierarchical Clustering*)** adopta un enfoque determinista fundamentalmente distinto: en lugar de buscar una única partición fija del espacio $\mathbb{R}^p$, construye una **familia anidada y continua de particiones** que describe las relaciones de proximidad a todas las escalas de distancia simultáneamente.

La taxonomía resultante se representa mediante un árbol binario enraizado denominado **Dendrograma**, donde la altura de cada unión vertical cuantifica exactamente la disimilaridad entre los subgrupos fusionados.

---

### Contenidos
1. Taxonomía del Clustering Jerárquico: Aglomerativo (*Bottom-Up*) vs. Divisivo (*Top-Down*)
2. Matriz de Disimilaridades y Espacios Métricos
3. El Algoritmo Aglomerativo Básico (Paso a Paso)
4. Concepto Geométrico del Dendrograma y Altura de Fusión
5. Distorsión Cofenética: Distancia Original vs. Distancia en el Árbol
6. Caso Práctico en Scikit-Learn y SciPy: Estructura de Mercados Financieros
7. Visualización del Primer Dendrograma Completo
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Aglomerativo (Bottom-Up) vs. Divisivo (Top-Down)

El agrupamiento jerárquico se divide en dos grandes filosofías:
- **Aglomerativo (*Bottom-Up*)**: Comienza con $n$ clusters individuales (cada observación es su propio grupo). En cada iteración, calcula las distancias entre todos los pares de clusters existentes y **fusiona los dos clusters más cercanos**, repitiendo hasta que todas las observaciones forman un único super-cluster en la raíz.
- **Divisivo (*Top-Down* / ej. DIANA)**: Comienza con un único cluster que contiene todo el dataset y recursivamente lo parte en dos hasta que cada punto queda aislado.

En la práctica industrial y de investigación, el enfoque **aglomerativo** es el estándar debido a su versatilidad métrica y menor complejidad computacional relativa ($O(n^2 \log n)$ frente a $O(2^n)$).


## 2. Construcción Paso a Paso del Dendrograma

Demostremos la mecánica exacta del algoritmo aglomerativo con un conjunto pequeño de puntos antes de pasar a datos de alta dimensionalidad.


In [ ]:
# Demostración ilustrativa: 6 activos financieros representados por volatilidad y retorno
activos = ['Bono_US', 'Bono_Aleman', 'Oro', 'Petroleo', 'Indice_SP500', 'Indice_NASDAQ']
X_toy = np.array([
    [0.5, 2.1],   # Bono_US
    [0.6, 2.0],   # Bono_Aleman
    [1.8, 4.5],   # Oro
    [2.9, 5.2],   # Petroleo
    [3.8, 9.5],   # Indice_SP500
    [4.2, 10.1]   # Indice_NASDAQ
])

# Matriz de distancias euclidianas por pares
D_toy = squareform(pdist(X_toy, metric='euclidean'))
df_dist_toy = pd.DataFrame(D_toy, index=activos, columns=activos)
print('Matriz de Distancias Euclidianas entre Activos:')
display(df_dist_toy.round(2))

# Matriz de enlace aglomerativo (Linkage Matrix)
Z_toy = linkage(X_toy, method='average', metric='euclidean')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Panel 1: Espacio de dispersión
axes[0].scatter(X_toy[:, 0], X_toy[:, 1], c='crimson', s=90, edgecolors='k')
for i, nombre in enumerate(activos):
    axes[0].annotate(nombre, (X_toy[i, 0] + 0.1, X_toy[i, 1] - 0.1), fontsize=10, fontweight='bold')
axes[0].set_title('Activos en el Espacio de Riesgo-Retorno', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Volatilidad Relativa')
axes[0].set_ylabel('Retorno Esperado (%)')

# Panel 2: Dendrograma asociado
dendrogram(Z_toy, labels=activos, orientation='right', ax=axes[1], color_threshold=0)
axes[1].set_title('Dendrograma de Fusión Aglomerativa', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Distancia de Disimilaridad (Altura)')
plt.tight_layout()
plt.show()


## 3. Matriz de Enlace (*Linkage Matrix*) en SciPy

En SciPy, el resultado de `linkage(X)` es una matriz de dimensiones $(n-1) \times 4$. Para $n$ observaciones, se ejecutan exactamente $n-1$ pasos de fusión. Cada fila $k$ contiene:
1. `Z[k, 0]` y `Z[k, 1]`: Los índices de los dos clusters que se fusionan en este paso (los clusters originales van de $0$ a $n-1$; los clusters recién formados reciben índices $n, n+1, \dots$).
2. `Z[k, 2]`: La distancia o altura a la cual se unieron.
3. `Z[k, 3]`: El número de observaciones originales contenidas en el nuevo cluster.

Revisemos la estructura exacta de la matriz de enlace de nuestro ejemplo:


In [ ]:
df_linkage_toy = pd.DataFrame(Z_toy, columns=['Cluster_A', 'Cluster_B', 'Distancia_Fusion', 'Muestras_Contenidas'])
df_linkage_toy['Cluster_Creado'] = [len(activos) + i for i in range(len(Z_toy))]
print('Matriz de Enlace (Linkage Matrix Z):')
display(df_linkage_toy[['Cluster_Creado', 'Cluster_A', 'Cluster_B', 'Distancia_Fusion', 'Muestras_Contenidas']].round(3))


## 4. Distorsión Cofenética (*Cophenetic Correlation*)

Cuando un algoritmo jerárquico resume una matriz de distancias multidimensional en un dendrograma, necesariamente introduce cierta distorsión. 

La **Distancia Cofenética** $c_{ij}$ entre dos observaciones $x_i$ y $x_j$ es la **altura vertical del nodo más bajo en el dendrograma donde $x_i$ y $x_j$ se unen por primera vez en el mismo cluster**.

El **Coeficiente de Correlación Cofenética (CPCC)** mide la correlación de Pearson lineal entre las distancias reales originales $d_{ij}$ y las distancias cofenéticas $c_{ij}$:

$$CPCC = \frac{\sum_{i < j} (d_{ij} - \bar{d})(c_{ij} - \bar{c})}{\sqrt{\sum_{i < j} (d_{ij} - \bar{d})^2 \sum_{i < j} (c_{ij} - \bar{c})^2}}$$

Un valor de $CPCC > 0.80$ indica que el dendrograma preserva fielmente las distancias métricas originales sin deformar la topología del dataset.


In [ ]:
# Cálculo del Coeficiente de Correlación Cofenética
c_coef, coph_dists = cophenet(Z_toy, pdist(X_toy))
print(f'Coeficiente de Correlación Cofenética (CPCC): {c_coef:.4f}')

plt.figure(figsize=(7, 5))
plt.scatter(pdist(X_toy), coph_dists, color='#1f77b4', s=60, edgecolors='k', alpha=0.8)
plt.plot([0, pdist(X_toy).max()], [0, pdist(X_toy).max()], 'r--', label='Ajuste Perfecto (Identidad)')
plt.title(f'Distancia Cofenética vs. Distancia Real (CPCC = {c_coef:.3f})', fontsize=11, fontweight='bold')
plt.xlabel('Distancia Euclidiana Real $d_{ij}$')
plt.ylabel('Distancia en el Dendrograma $c_{ij}$')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Implementación en Scikit-Learn: `AgglomerativeClustering`

Scikit-Learn proporciona la clase `sklearn.cluster.AgglomerativeClustering`, diseñada para integrarse directamente en pipelines productivos con `fit_predict`.

A diferencia de SciPy (que construye todo el árbol completo para visualización), Scikit-Learn optimiza la memoria calculando el agrupamiento podado a un número de clusters `n_clusters` o a un umbral `distance_threshold`.


In [ ]:
# Ajuste en Scikit-Learn para cortar el árbol en 3 clusters
agg_model = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='average')
etiquetas = agg_model.fit_predict(X_toy)

df_clusters = pd.DataFrame({'Activo': activos, 'Cluster': etiquetas})
print('Asignación de Clusters con AgglomerativeClustering (n_clusters=3):')
display(df_clusters)


## 6. Conclusiones

1. **Estructura Jerárquica Continua**: El clustering aglomerativo no obliga a comprometerse con un valor arbitrario de $K$ antes de entrenar; preserva toda la jerarquía de fusión.
2. **Dendrograma**: Es una herramienta visual determinante para dominios que requieren auditoría y explicabilidad de relaciones de parentesco o proximidad entre entidades.
3. **Coeficiente Cofenético**: Permite validar matemáticamente si el árbol representa honestamente las distancias originales.

En el siguiente cuaderno, **02_Criterios_de_Enlace_y_Validacion_Cofenetica.ipynb**, estudiaremos cómo la definición de distancia entre clusters (*Single, Complete, Average, Ward*) cambia radicalmente la forma y calidad de los grupos resultantes.
